# 4. Phylogeny
Since we are using 16S rRNA, we will use a refernce-based method to build our phylogenetic tree. In particular we will use fragment insertion which places short query sequences (e.g., ASVs) into a high-quality existing reference tree using SATé-enabled phylogenetic placement (SEPP). Our reference tree is built from the SILVA 13_8 database.
SILVA Reference alignments are particularly powerful for rRNA gene sequence data, as knowledge of secondary structure is incorporated into the curation process, thus increasing alignment quality.
This will give us accurate UniFrac distances without reconstructing everything from scratch.

In [4]:
# 1- Import packages
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt
import numpy as np
import qiime2 as q2
%matplotlib inline

In [5]:
# 2 - Set working directory
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/Project/MicrobiomeAnalysis_TummyTribe/scripts").


In [6]:
# 3 - Data directories
raw_data_dir = "../data/raw"
phylogeny_data_dir = "../data/processed/phylogeny"
denoising_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"
metadata_dir = "../data/raw"

# Fragment insertion

We can get the SILVA 128 reference tree from qiime. According to a user in the qiime forum using SILVA 138 for the taxonomy and SILVA 128 to build the phylogenetic tree should not lead to any complications (https://forum.qiime2.org/t/compatibility-of-sepp-silva128-with-taxonomy-classification-using-silva138/31172?utm_source=chatgpt.com).

In [18]:
#SILVA 128 for SEPP
! wget -O $phylogeny_data_dir/silva-128-sepp-refs.qza https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza

--2025-11-11 13:28:36--  https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza [following]
--2025-11-11 13:28:36--  https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.218.246.176, 52.92.162.192, 52.218.236.160, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.218.246.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 181253322 (173M) [binary/octet-stream]
Saving to: ‘../data/processed/phylogeny/silva-128-sepp-refs.qza’

../data/processed/p 100%[===================>] 172.86M  16.3MB/s    in 12s     

2025-11-11

We can quickly check if the our sequence file and the reference database have the right format and are compatible.

In [19]:
!qiime tools peek $denoising_data_dir/dada2_rep_seq.qza 

UUID:        0224622c-56b2-43d5-8034-c3d398690a51
Type:        FeatureData[Sequence]
Data format: DNASequencesDirectoryFormat


In [22]:
!qiime tools peek $phylogeny_data_dir/silva-128-sepp-refs.qza

UUID:        e44b5e78-31e5-4a0f-9041-494bc3ca2df2
Type:        SeppReferenceDatabase
Data format: SeppReferenceDirFmt


## Fragment insertion using SEPP
The following cell does not run in this notebook due to memory issues. It was run on euler and the finished tree and tree placements can be imported for visualization and further analysis.

In [21]:
#does not run in notebook -> euler (needs amplicon env)
! qiime fragment-insertion sepp \
    --i-representative-sequences $denoising_data_dir/dada2_rep_seq.qza \
    --i-reference-database $phylogeny_data_dir/silva-128-sepp-refs.qza \
    --p-threads 2 \
    --o-tree $phylogeny_data_dir/sepp-tree.qza \
    --o-placements $phylogeny_data_dir/sepp-tree-placements.qza \
    --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Removing /tmp/tmp.1ZZbpa3ZS5/sepp-tmp-QfUBleakIt


Check if outpoot tree has expected format. It's already rooted and has Newick tree format.

In [11]:
!qiime tools peek  $phylogeny_data_dir/sepp-tree.qza

UUID:        2668cfca-0322-462b-a652-2a9dca8a799f
Type:        Phylogeny[Rooted]
Data format: NewickDirectoryFormat


## Tree visualization

In [7]:
! qiime empress tree-plot \
    --i-tree $phylogeny_data_dir/sepp-tree.qza \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $data_dir/sepp-tree.qzv

Error: QIIME 2 has no plugin/command named 'empress'.


Export tree, taxonomy information and feature table for tree visualization. Visualization was done in R. The script is available in the additional scripts folder.

In [14]:
#export newick tree and the tree placements
! qiime tools export \
    --input-path $phylogeny_data_dir/sepp-tree.qza \
    --output-path $phylogeny_data_dir/exported-tree

! qiime tools export \
    --input-path $phylogeny_data_dir/sepp-tree-placements.qza \
    --output-path $phylogeny_data_dir/exported-placements


/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/phylogeny/sepp-tree.qza as NewickDirectoryFormat to directory ../data/processed/phylogeny/exported-tree
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/phylogeny/sepp-tree-placements.qza as PlacementsDirFmt to directory ../data/processed/phylogeny/exported-placements


In [ ]:
#export taxonomy
! qiime tools export \
    --input-path $taxonomy_data_dir/taxonomy.qza \
    --output-path $taxonomy_data_dir/taxonomy_export

In [ ]:
#export feature table
! qiime tools export \
    --input-path $denoising_data_dir/dada2_table.qza \
    --output-path $denoising_data_dir/exported-feature-table

In [6]:
! biom convert \
    -i $denoising_data_dir/exported-feature-table/feature-table.biom \
    -o $denoising_data_dir/exported-feature-table/feature-table.tsv \
    --to-tsv

# De novo tree construction
As an alternative to fragment insertion we can also create a phylogenetic tree by aligning the marker genes across divergent taxa and try to reconstruct the tree based on the resulting alignment. One of the issues of this approach is that short sequences may not carry enough information to capture a meaningful phylogeny.

## Sequence alignemnt

In [10]:
! qiime alignment mafft \
    --i-sequences $denoising_data_dir/dada2_rep_seq.qza \
    --o-alignment $phylogeny_data_dir/aligned-rep-seqs.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[AlignedSequence] to: ../data/processed/phylogeny/aligned-rep-seqs.qza


## Alignemnt masking

In [11]:
! qiime alignment mask \
    --i-alignment $phylogeny_data_dir/aligned-rep-seqs.qza \
    --o-masked-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[AlignedSequence] to: ../data/processed/phylogeny/masked-aligned-rep-seqs.qza


## Tree construction

In [12]:
! qiime phylogeny fasttree \
    --i-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza \
    --o-tree $phylogeny_data_dir/fasttree-tree.qza

! qiime phylogeny midpoint-root \
    --i-tree $phylogeny_data_dir/fasttree-tree.qza \
    --o-rooted-tree $phylogeny_data_dir/fasttree-tree-rooted.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Unrooted] to: ../data/processed/phylogeny/fasttree-tree.qza
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Rooted] to: ../data/processed/phylogeny/fasttree-tree-rooted.qza


In [15]:
#export newick tree and the tree placements
! qiime tools export \
    --input-path $phylogeny_data_dir/fasttree-tree-rooted.qza \
    --output-path $phylogeny_data_dir/exported-denovo-tree

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/phylogeny/fasttree-tree-rooted.qza as NewickDirectoryFormat to directory ../data/processed/phylogeny/exported-denovo-tree


## Tree visualization

In [13]:
! qiime empress tree-plot \
    --i-tree $phylogeny_data_dir/fasttree-tree-rooted.qza \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $phylogeny_data_dir/fasttree-tree-rooted.qzv

Error: QIIME 2 has no plugin/command named 'empress'.
